In [4]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import os
import glob

# ==========================================
# 1. SETUP AND CONFIGURATION
# ==========================================
data_dir = r'C:\CRSS_ABM\CRSS2023\results\ScientificReports'

# Global Font Settings for Publication
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.size'] = 12
plt.rcParams['axes.labelsize'] = 14
plt.rcParams['axes.titlesize'] = 15
plt.rcParams['xtick.labelsize'] = 12
plt.rcParams['ytick.labelsize'] = 12
plt.rcParams['legend.fontsize'] = 11

# Aesthetic Color Palette
color_outer = '#A9CCE3'  
color_inner = '#2980B9'  
color_trace = '#7F8C8D'  
color_ref_1 = '#C0392B'  
color_ref_2 = '#E67E22'  

csv_files = glob.glob(os.path.join(data_dir, '*.csv'))

if not csv_files:
    print("No CSV files found in the specified directory.")
else:
    print(f"Found {len(csv_files)} files. Starting batch processing...")

# ==========================================
# 2. BATCH PROCESSING LOOP
# ==========================================
for file_path in csv_files:
    alt_name = os.path.splitext(os.path.basename(file_path))[0]
    print(f"\nProcessing Alternative: {alt_name}...")
    
    df = pd.read_csv(file_path)
    df['Timestep'] = pd.to_datetime(df['Timestep'])
    
    df_elev = df[df['Object.Slot'].isin(['Powell.Pool Elevation', 'Mead.Pool Elevation'])].copy()
    df_elev['Reservoir Name'] = df_elev['Object.Slot'].apply(lambda x: x.split('.')[0])

    # ==========================================
    # 3. PLOTTING FUNCTION FOR CURRENT ALTERNATIVE
    # ==========================================
    fig, axes = plt.subplots(1, 2, figsize=(15, 6.5))
    reservoirs = ['Powell', 'Mead']

    for i, res in enumerate(reservoirs):
        ax = axes[i]
        
        res_data = df_elev[df_elev['Reservoir Name'] == res]
        pivot_data = res_data.pivot_table(index='Timestep', columns='Trace Number', values='Slot Value')
        
        q10 = pivot_data.quantile(0.10, axis=1)
        q25 = pivot_data.quantile(0.25, axis=1)
        q75 = pivot_data.quantile(0.75, axis=1)
        q90 = pivot_data.quantile(0.90, axis=1)
        
        # Plot Uncertainty Bands and Traces
        ax.fill_between(pivot_data.index, q10, q90, color=color_outer, alpha=0.6, zorder=1)
        ax.fill_between(pivot_data.index, q25, q75, color=color_inner, alpha=0.7, zorder=2)
        ax.plot(pivot_data.index, pivot_data.values, color=color_trace, alpha=0.15, linewidth=0.5, zorder=3)
        
        # ---------------------------------------------------------
        # THRESHOLD LINES & STANDARDIZED Y-AXIS SCALING
        # ---------------------------------------------------------
        if res == 'Powell':
            ax.axhline(y=3575, color=color_ref_1, linestyle='--', linewidth=1.5, zorder=4)
            ax.axhline(y=3490, color=color_ref_2, linestyle=':', linewidth=2, zorder=4)
            
            # Lock Powell Axis: Bottom at 3375, Top at 3715 (to give breathing room above 3700)
            ax.set_ylim(3375, 3715)
            # Force labels only for 3400 to 3700 in 50ft intervals
            ax.set_yticks(list(range(3400, 3701, 50)))
            
        elif res == 'Mead':
            ax.axhline(y=1075, color=color_ref_1, linestyle='--', linewidth=1.5, zorder=4)
            ax.axhline(y=950, color=color_ref_2, linestyle=':', linewidth=2, zorder=4)
            
            # Lock Mead Axis: Standardized comparable range
            ax.set_ylim(900, 1250)
            # Force labels from 900 to 1250 in 50ft intervals
            ax.set_yticks(list(range(900, 1251, 50)))

        # Formatting & Aesthetics
        ax.set_title(f'Lake {res} Elevation Uncertainty (ft) - {alt_name}', pad=10)
        ax.set_ylabel('Elevation (ft)')
        
        # Set X-axis limits and 5-year ticks
        ax.set_xlim(pd.to_datetime('2024-01-01'), pd.to_datetime('2060-12-31'))
        ax.xaxis.set_major_locator(mdates.YearLocator(5))
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
        
        ax.grid(True, axis='y', linestyle='-', color='#E5E7E9', alpha=0.7, zorder=0)

        # Custom legend on the first panel
        if i == 0:
            from matplotlib.patches import Patch
            from matplotlib.lines import Line2D
            legend_elements = [
                Patch(facecolor=color_inner, alpha=0.7, label='25th - 75th Percentile'),
                Patch(facecolor=color_outer, alpha=0.6, label='10th - 90th Percentile'),
                Line2D([0], [0], color=color_trace, lw=1, alpha=0.8, label='Individual Traces'),
                Line2D([0], [0], color=color_ref_1, lw=1.5, linestyle='--', label='Primary Thresholds (3575/1075 ft)'),
                Line2D([0], [0], color=color_ref_2, lw=2, linestyle=':', label='Critical Thresholds (3490/950 ft)')
            ]
            ax.legend(handles=legend_elements, loc='lower left', framealpha=0.9, edgecolor='#BDC3C7')

    plt.tight_layout()

    # PNG EXPORT
    output_filename = os.path.join(data_dir, f'Figure_S1_{alt_name}_Uncertainty.png')
    plt.savefig(output_filename, dpi=600, bbox_inches='tight', transparent=False)
    print(f"Saved: {output_filename}")
    
    plt.close(fig)

print("\nAll alternatives processed successfully!")

Found 5 files. Starting batch processing...

Processing Alternative: BHA...
Saved: C:\CRSS_ABM\CRSS2023\results\ScientificReports\Figure_S1_BHA_Uncertainty.png

Processing Alternative: CCA...
Saved: C:\CRSS_ABM\CRSS2023\results\ScientificReports\Figure_S1_CCA_Uncertainty.png

Processing Alternative: FAA...
Saved: C:\CRSS_ABM\CRSS2023\results\ScientificReports\Figure_S1_FAA_Uncertainty.png

Processing Alternative: FAHA...
Saved: C:\CRSS_ABM\CRSS2023\results\ScientificReports\Figure_S1_FAHA_Uncertainty.png

Processing Alternative: NAA...
Saved: C:\CRSS_ABM\CRSS2023\results\ScientificReports\Figure_S1_NAA_Uncertainty.png

All alternatives processed successfully!
